# Continue a fine-tune on Colab (free T4) (notebook v11)

**Notebook version v11.** Changes:
* **v11** (2026-11-10): round 8 (106 topics: round 7 plus deeper answers). Starts from omni-cv/ckpt_v7; patience 8; keeps the best weights in omni-cv/ckpt_v8 and the last ones in ckpt_v8/last. Needs the zip of 1,299,606,077 bytes.
* v10 (2026-11-07): starts from omni-cv/ckpt_v3, patched answers (no email, new introduction and disclaimer); result in omni-cv/ckpt_v4. Needs the zip of 1,091,590,584 bytes.
* v9 (2026-11-06): starts from omni-cv/ckpt_v2; early stopping on a validation split (5% of the training wordings), up to 6 epochs, keeps the best; result in omni-cv/ckpt_v3. Needs the patched zip (1,092,080,226 bytes).
* v8 (2026-11-06): round 6 (68 topics, RAG); settings from the GPU (A100: bf16, batch 8); 1 epoch at lr 1e-5.
* v4 (2026-11-06): continues from `omni-cv/ckpt` on the round-4 data (40 topics), `--typed --cv-weight 1`,
  local folder `/content/omni-cv`.
* v2 (2026-11-05): 1 epoch at lr 5e-6. v1: first version.

Starts from a finished run (`omni-cv/ckpt`) and trains on updated data, for example after adding Q&A topics.
It trains on **all** the data, old and new: training on the new topics alone would make the model forget the rest.

1. Upload the new bundle as `omni-cv/omni_cv_train.zip` (keep `omni-cv/ckpt` from the first run).
2. **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. If the browser disconnects, Run all again: it resumes while the same Colab machine is alive.
3. When it finishes, download `omni-cv/ckpt_v8` (best weights, plus `last/`).

In [ ]:
NOTEBOOK_VERSION = 'v11'
EXPECTED_TOPICS = 106  # round 8
print('notebook', NOTEBOOK_VERSION)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/omni-cv'
import os
assert os.path.exists(f'{D}/ckpt_v7/lit_model.pth'), 'omni-cv/ckpt_v7 (the starting point) is missing'
size = os.path.getsize(f'{D}/omni_cv_train.zip')
print('zip', size, 'bytes')
assert size == 1299606077, 'not the round-8 zip (expected 1,299,606,077 bytes): wait for the upload, or upload the new one'

In [ ]:
!rm -rf /content/omni-cv && mkdir -p /content/omni-cv && cd /content/omni-cv && unzip -q $D/omni_cv_train.zip && ls /content/omni-cv/finetune/work/encoded

In [ ]:
import torch
ev = torch.load('/content/omni-cv/finetune/work/encoded/eval.pt', weights_only=False)
topics = {r['id'] for r in ev if r['kind'] == 'cv'}
print(f'data: {len(ev)} held-out examples, {len(topics)} CV topics')
assert len(topics) == EXPECTED_TOPICS, f'wrong zip: {len(topics)} topics, expected {EXPECTED_TOPICS}'
assert all('ctx' in r for r in ev), 'wrong zip: no retrieved context (RAG) in it'
print('RAG context: yes')

In [ ]:
from huggingface_hub import hf_hub_download
for f in ['model_config.yaml', 'tokenizer.json', 'tokenizer_config.json']:  # the weights come from omni-cv/ckpt
    hf_hub_download('gpt-omni/mini-omni2', f, local_dir='/content/mini-omni2')

Training. Progress is saved every 200 steps to Colab's own disk (`/content`), not to Drive: on a mounted Drive,
every replaced file goes to the Trash and still counts against your storage (6.4 GB per save). If the browser
disconnects, reconnect and Run all: it resumes while the same Colab machine is alive. Only the result goes to
Drive (~3 GB).

In [ ]:
# ---- settings: filled in from the GPU, change them here if you like ----
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
AMP = 'bf16' if torch.cuda.get_device_capability(0)[0] >= 8 else 'fp16'
BATCH, ACCUM = (8, 1) if mem >= 20 else (2, 4)
LR, EPOCHS = 1e-5, 6            # up to 6 epochs: early stopping usually ends it sooner
VAL_FRAC, VAL_EVERY, PATIENCE = 0.05, 200, 8
print(f'--amp {AMP} --batch {BATCH} --lr {LR} --epochs {EPOCHS}, early stopping every {VAL_EVERY} steps, patience {PATIENCE}; from {D}/ckpt_v7')
%cd /content/omni-cv/finetune
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MINI_OMNI2_DIR=/content/mini-omni2 FT_WORK=/content/omni-cv/finetune/work python 04_train.py \
    --init $D/ckpt_v7/lit_model.pth --amp $AMP --batch $BATCH --accum $ACCUM --epochs $EPOCHS --lr $LR --cv-weight 1 --typed \
    --val-frac $VAL_FRAC --val-every $VAL_EVERY --patience $PATIENCE --save-last \
    --state /content/state_v8.pt --save-every 200 --out $D/ckpt_v8

In [ ]:
!ls -la $D/ckpt_v8 $D/ckpt_v8/last